In [1]:
# Load skincare products with KNN predictions and K-Means clusters.
import pandas as pd
products = pd.read_csv("skincare_with_predictions.csv")
clusters = pd.read_csv("skincare_clustered.csv")
print("Total Products:", len(products))
print("Total Clustered Products:", len(clusters))
print("Number of Clusters:", clusters["Cluster"].nunique())

print("\nSame Product Order:", products[["brand", "name", "ingredients_clean"]].equals(clusters[["brand", "name", "ingredients_clean"]] ))


Total Products: 2724
Total Clustered Products: 2724
Number of Clusters: 9

Same Product Order: True


In [2]:
# Combine KNN predictions and K-Means cluster numbers into one dataset.
if products[["brand", "name", "ingredients_clean"]].equals( clusters[["brand", "name", "ingredients_clean"]]):
    products["Cluster"] = clusters["Cluster"].values

    print("Datasets Combined Successfully!")
    print("Total Products:", len(products))
    print("Total Clusters:", products["Cluster"].nunique())

    display(products[["brand", "name", "category", "price_pkr", "Cluster"]].head())
else:
    print("Product order does not match. Datasets were not combined.")


Datasets Combined Successfully!
Total Products: 2724
Total Clusters: 9


,brand,name,category,price_pkr,Cluster
0,The Ordinary,Balancing & Clarifying Serum,Serum,6624.0,2
1,The Ordinary,GF 15% Solution,Serum,5155.0,2
2,The Ordinary,Multi-Antioxidant Radiance Serum,Treatment,4656.0,2
3,The Ordinary,Natural Moisturizing Factors + PhytoCeramides,Moisturizer,3603.0,4
4,The Ordinary,Retinal 0.2% Emulsion,Serum,4961.0,2


In [3]:
# Combine original and predicted skin-type labels for product recommendations.
skin_cols = ["tag_dry", "tag_oily", "tag_combination", "tag_normal", "tag_sensitive", "tag_acne_prone"]
for skin in skin_cols:
    new_col = "match_" + skin

    products[new_col] = products[skin].fillna(0).astype(int)

    products.loc[products["has_skin_label"] == 0, new_col] = products.loc[products["has_skin_label"] == 0, "pred_" + skin ].astype(int)

print("Skin-Type Labels Prepared Successfully!")

match_cols = ["match_" + skin for skin in skin_cols]

print("\nProducts Matching Each Skin Type:")
print(products[match_cols].sum())


Skin-Type Labels Prepared Successfully!

Products Matching Each Skin Type:
match_tag_dry            1968
match_tag_oily           1696
match_tag_combination    2295
match_tag_normal         2335
match_tag_sensitive       822
match_tag_acne_prone      510
dtype: int64


In [4]:
# Filter skincare products according to skin type, category, and budget.
print("Available Categories:")
print(products["category"].dropna().unique())
skin_type = input("\nEnter skin type: ").strip().lower()
category = input("Enter category (or All): ").strip()
budget = float(input("Enter maximum budget in PKR: "))
skin_column = "match_tag_" + skin_type
if skin_column in products.columns:
    filtered_products = products[
        (products[skin_column] == 1) &
        (products["price_pkr"] <= budget)].copy()

    if category.lower() != "all":
        filtered_products = filtered_products[
            filtered_products["category"].str.lower() == category.lower()
        ]

    print("\nMatching Products:", len(filtered_products))

    display(filtered_products[
        ["brand", "name", "category", "price_pkr", "Cluster"]
    ].head(10))

else:
    print("Invalid skin type!")


Available Categories:
['Serum' 'Treatment' 'Moisturizer' 'Cleanser' 'Sun protection' 'Eye care'
 'Mask']



Enter skin type:  oily
Enter category (or All):  serum
Enter maximum budget in PKR:  5000



Matching Products: 35


,brand,name,category,price_pkr,Cluster
4,The Ordinary,Retinal 0.2% Emulsion,Serum,4961.0,2
5,The Ordinary,Amino Acids + B5,Serum,3049.0,2
6,The Ordinary,Niacinamide 10% + Zinc 1%,Serum,1996.0,2
9,The Ordinary,Granactive Retinoid 2% Emulsion,Serum,4019.0,2
10,The Ordinary,Caffeine 3% + Escin 1% Face Serum,Serum,2993.0,2
13,The Ordinary,Alpha Arbutin 2% + HA,Serum,3825.0,4
16,The Ordinary,Ascorbyl Glucoside Solution 12%,Serum,4934.0,2
20,The Ordinary,Retinol 0.3% Emulsion,Serum,3160.0,2
21,The Ordinary,Retinol 1% in Squalane,Serum,3298.0,3
31,The Ordinary,Argireline Solution 10%,Serum,3215.0,2


In [5]:
# Recommend 5 products with ingredients similar to a selected product.
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity
X_tfidf = load_npz("ingredients_tfidf.npz")
product_index = int(input("Enter a product index from the table: "))
if product_index in filtered_products.index:
    similarity = cosine_similarity( X_tfidf[product_index], X_tfidf[filtered_products.index])[0]
    recommendations = filtered_products.copy()
    recommendations["Similarity"] = similarity
    recommendations = recommendations.drop(product_index)
    recommendations = recommendations.sort_values("Similarity", ascending=False )
    print("\nTop 5 Similar Products:")
    display(recommendations[
        ["brand", "name", "category", "price_pkr", "Similarity", "Cluster"]].head(5))
else:
    print("Invalid product index!")


Enter a product index from the table:  6



Top 5 Similar Products:


,brand,name,category,price_pkr,Similarity,Cluster
16,The Ordinary,Ascorbyl Glucoside Solution 12%,Serum,4934.0,0.468003,2
31,The Ordinary,Argireline Solution 10%,Serum,3215.0,0.380161,2
36,The Ordinary,Hyaluronic Acid 2% + B5 (Original Formulation),Serum,2966.0,0.343693,4
1651,The Ordinary,Granactive Retinoid* 2% Emulsion,Serum,3243.0,0.208442,2
1650,The Ordinary,Alpha Arbutin 2% + HA Hyperpigmentation Serum,Serum,3077.0,0.205213,4


In [6]:
# Display common ingredients between the selected product and its recommendations.
selected_ingredients = set(products.loc[product_index, "ingredients_clean"].split("|"))
print("Selected Product:", products.loc[product_index, "name"])
for i, row in recommendations.head(5).iterrows():
    other_ingredients = set(row["ingredients_clean"].split("|"))
    common = selected_ingredients.intersection(other_ingredients)
    print("\nRecommended Product:", row["name"])
    print("Common Ingredients:", len(common))
    print(", ".join(sorted(common)[:10]))


Selected Product: Niacinamide 10% + Zinc 1%

Recommended Product: Ascorbyl Glucoside Solution 12%
Common Ingredients: 5
dimethyl isosorbide, ethoxydiglycol, isoceteth-20, water, xanthan gum

Recommended Product: Argireline Solution 10%
Common Ingredients: 5
chlorphenesin, dimethyl isosorbide, isoceteth-20, phenoxyethanol, water

Recommended Product: Hyaluronic Acid 2% + B5 (Original Formulation)
Common Ingredients: 5
ethoxydiglycol, isoceteth-20, pentylene glycol, phenoxyethanol, water

Recommended Product: Granactive Retinoid* 2% Emulsion
Common Ingredients: 6
chlorphenesin, dimethyl isosorbide, isoceteth-20, phenoxyethanol, water, xanthan gum

Recommended Product: Alpha Arbutin 2% + HA Hyperpigmentation Serum
Common Ingredients: 4
chlorphenesin, ethoxydiglycol, phenoxyethanol, water


In [7]:
# Check the selected product's important ingredients and compare them with recommendations.
active_cols = [
    "has_niacinamide",
    "has_zinc",
    "has_salicylic_acid",
    "has_ceramide",
    "has_hyaluronic",
    "has_squalane"
]

print("Selected Product:", products.loc[product_index, "name"])

print("\nSelected Product's Ingredient Features:")
print(products.loc[product_index, active_cols])

print("\nRecommended Products' Ingredient Features:")
display(recommendations[["name"] + active_cols].head(5))


Selected Product: Niacinamide 10% + Zinc 1%

Selected Product's Ingredient Features:
has_niacinamide       1
has_zinc              1
has_salicylic_acid    0
has_ceramide          0
has_hyaluronic        0
has_squalane          0
Name: 6, dtype: object

Recommended Products' Ingredient Features:


,name,has_niacinamide,has_zinc,has_salicylic_acid,has_ceramide,has_hyaluronic,has_squalane
16,Ascorbyl Glucoside Solution 12%,0,0,0,0,0,0
31,Argireline Solution 10%,0,0,0,0,0,0
36,Hyaluronic Acid 2% + B5 (Original Formulation),0,0,0,0,1,0
1651,Granactive Retinoid* 2% Emulsion,0,0,0,0,0,0
1650,Alpha Arbutin 2% + HA Hyperpigmentation Serum,0,0,0,0,1,0


In [8]:
# Rank products by shared active ingredients and then by cosine similarity.
selected_actives = []
for col in active_cols:
    if products.loc[product_index, col] == 1:
        selected_actives.append(col)

print("Selected Product's Important Ingredients:")
print(selected_actives)

recommendations["Shared_Actives"] = recommendations[selected_actives].sum(axis=1)

recommendations = recommendations.sort_values(
    ["Shared_Actives", "Similarity"],
    ascending=[False, False]
)

print("\nTop 5 Improved Recommendations:")

display(recommendations[
    ["brand", "name", "price_pkr", "Shared_Actives", "Similarity", "Cluster"]
].head(5))


Selected Product's Important Ingredients:
['has_niacinamide', 'has_zinc']

Top 5 Improved Recommendations:


,brand,name,price_pkr,Shared_Actives,Similarity,Cluster
1640,The INKEY List,SuperSolutions 20% Niacinamide Serum,4986.0,2,0.202152,3
86,Blissful,Acne Serum,2200.0,1,0.144364,2
1614,The INKEY List,Niacinamide Oil Control Serum,2769.0,1,0.070763,5
16,The Ordinary,Ascorbyl Glucoside Solution 12%,4934.0,0,0.468003,2
31,The Ordinary,Argireline Solution 10%,3215.0,0,0.380161,2


In [9]:

# Show alternatives that share at least one important ingredient with the selected product.

active_matches = recommendations[
    recommendations["Shared_Actives"] > 0
].copy()

print("Products Sharing Important Ingredients:", len(active_matches))

if len(active_matches) > 0:
    print("\nTop 5 Active-Ingredient Alternatives:")

    display(active_matches[
        ["brand", "name", "price_pkr", "Shared_Actives", "Similarity", "Cluster"]
    ].head(5))

else:
    print("No matching active-ingredient alternatives found within the selected filters.")


Products Sharing Important Ingredients: 3

Top 5 Active-Ingredient Alternatives:


,brand,name,price_pkr,Shared_Actives,Similarity,Cluster
1640,The INKEY List,SuperSolutions 20% Niacinamide Serum,4986.0,2,0.202152,3
86,Blissful,Acne Serum,2200.0,1,0.144364,2
1614,The INKEY List,Niacinamide Oil Control Serum,2769.0,1,0.070763,5


In [10]:
# Check niacinamide and zinc ingredients in the selected product and its alternatives.
check_products = products.loc[
    [product_index] + active_matches.head(5).index.tolist()
]

for i, row in check_products.iterrows():

    print("\nProduct:", row["name"])

    print("Niacinamide Flag:", row["has_niacinamide"])
    print("Zinc Flag:", row["has_zinc"])

    print("Matching Ingredients:")

    for ingredient in row["ingredients_clean"].split("|"):
        if "niacinamide" in ingredient or "zinc" in ingredient:
            print(ingredient)



Product: Niacinamide 10% + Zinc 1%
Niacinamide Flag: 1
Zinc Flag: 1
Matching Ingredients:
niacinamide
zinc pca

Product: SuperSolutions 20% Niacinamide Serum
Niacinamide Flag: 1
Zinc Flag: 1
Matching Ingredients:
niacinamide
zinc pca

Product: Acne Serum
Niacinamide Flag: 1
Zinc Flag: 0
Matching Ingredients:
niacinamide

Product: Niacinamide Oil Control Serum
Niacinamide Flag: 1
Zinc Flag: 0
Matching Ingredients:
niacinamide


In [22]:

# Improve recommendations by allowing general alternatives and filtering sunscreen names.

def recommend_products(product_index, skin_type, category, budget):

    skin_column = "match_tag_" + skin_type.lower()

    if skin_column not in products.columns or product_index not in products.index:
        print("Invalid skin type or product index!")
        return pd.DataFrame()

    matches = products[
        (products[skin_column] == 1) &
        (products["price_pkr"] <= budget)
    ].copy()

    if category.lower() != "all":
        matches = matches[
            matches["category"].str.lower() == category.lower()
        ]

    if category.lower() == "sun protection":
        matches = matches[
            matches["name"].str.contains(
                "SPF|sunscreen|sunblock|broad spectrum",
                case=False, na=False
            )
        ]

    if product_index in matches.index:
        matches = matches.drop(product_index)

    if len(matches) == 0:
        print("No products found within the selected filters.")
        return pd.DataFrame()

    matches["Similarity"] = cosine_similarity(
        X_tfidf[product_index],
        X_tfidf[matches.index]
    )[0]

    selected_actives = []

    for col in active_cols:
        if products.loc[product_index, col] == 1:
            selected_actives.append(col)

    if len(selected_actives) > 0:
        matches["Shared_Actives"] = matches[selected_actives].sum(axis=1)
    else:
        matches["Shared_Actives"] = 0

    matches["Match_Type"] = "General ingredient similarity"

    matches.loc[
        matches["Shared_Actives"] > 0, "Match_Type"
    ] = "Shared key ingredients"
    
# Remove products that share no important ingredients and have zero similarity.

    matches = matches[
    (matches["Shared_Actives"] > 0) |
    (matches["Similarity"] > 0)
    ]

    if len(matches) == 0:
        print("No ingredient-similar alternatives found.")
        return pd.DataFrame()

    matches = matches.sort_values(
        ["Shared_Actives", "Similarity"],
        ascending=[False, False]
    )

    return matches[
        ["brand", "name", "category", "price_pkr",
         "Shared_Actives", "Similarity", "Match_Type", "Cluster"]
    ].head(5)


# Test the previously empty sunscreen recommendations.

results3 = recommend_products(3, "combination", "Sun protection", 6000)

display(results3)


,brand,name,category,price_pkr,Shared_Actives,Similarity,Match_Type,Cluster
2606,Supergoop!,Everyday Sunscreen Broad Spectrum SPF 50,Sun protection,5266.0,0,0.098759,General ingredient similarity,1
2712,Mdsolarsciences,Quick Dry Body Spray With SolSci-X™ Broad Spec...,Sun protection,5543.0,0,0.021781,General ingredient similarity,2
816,innisfree,Daily UV Defense Invisible Broad Spectrum SPF ...,Sun protection,4435.0,0,0.020729,General ingredient similarity,6
2697,Mdsolarsciences,Mineral Sunscreen Stick Broad Spectrum SPF 40 ...,Sun protection,4989.0,0,0.012111,General ingredient similarity,3


In [12]:

# Test the recommendation function with a different product and skin type.

print("Selected Product:", products.loc[3, "name"])

results2 = recommend_products(3, "dry", "Moisturizer", 6000)

print("\nRecommended Products:")
display(results2)


Selected Product: Natural Moisturizing Factors + PhytoCeramides

Recommended Products:


,brand,name,category,price_pkr,Shared_Actives,Similarity,Cluster
12,The Ordinary,Natural Moisturizing Factors + HA,Moisturizer,2217.0,1,0.584087,4
8,The Ordinary,Rice Lipids + Ectoin Microemulsion,Moisturizer,4352.0,1,0.561720,4
1331,Sephora Collection,All Day Hyaluronic Acid Hydrator Moisturizer,Moisturizer,4989.0,1,0.093011,4
833,IT Cosmetics,Confidence in a Cream Anti-Aging Hydrating Moi...,Moisturizer,5543.0,1,0.079759,5
72,CeraVe,Ultra-Light Moisturizing Gel,Moisturizer,5541.0,1,0.075420,4


In [14]:

# Test recommendations for different skin types and product categories.

print("Sensitive Skin - Cleanser")
display(recommend_products(6, "sensitive", "Cleanser", 6000))

print("\nAcne-Prone Skin - Treatment")
display(recommend_products(6, "acne_prone", "Treatment", 6000))

print("\nCombination Skin - Sun Protection")
display(recommend_products(3, "combination", "Sun protection", 6000))


Sensitive Skin - Cleanser


,brand,name,category,price_pkr,Shared_Actives,Similarity,Cluster
81,CeraVe,Hydrating Toner,Cleanser,3323.0,1,0.097824,4
50,CeraVe,Hydrating Micellar Water,Cleanser,4155.0,1,0.063517,2



Acne-Prone Skin - Treatment


,brand,name,category,price_pkr,Shared_Actives,Similarity,Cluster
1369,Sephora Collection,Clarifying Toner with Niacinamide and Zinc,Treatment,3880.0,2,0.098390,2
19,The Ordinary,Sulfur 10% Powder-to-Cream Concentrate,Treatment,3160.0,1,0.103627,2
78,CeraVe,Acne Control Gel,Treatment,5818.0,1,0.101096,4
1052,Mario Badescu,Buffering Lotion,Treatment,5266.0,1,0.084661,2
1377,Sephora Collection,Acne Treatment Gel,Treatment,5543.0,1,0.058859,2



Combination Skin - Sun Protection


,brand,name,category,price_pkr,Shared_Actives,Similarity,Cluster


In [15]:

# Check how many sunscreens match the skin type and budget before ingredient filtering.

sun_matches = products[
    (products["match_tag_combination"] == 1) &
    (products["category"].str.lower() == "sun protection") &
    (products["price_pkr"] <= 6000)
]

print("Matching Sunscreens Before Ingredient Filtering:", len(sun_matches))

display(sun_matches[
    ["brand", "name", "category", "price_pkr", "Cluster"]
].head(10))


Matching Sunscreens Before Ingredient Filtering: 8


,brand,name,category,price_pkr,Cluster
816,innisfree,Daily UV Defense Invisible Broad Spectrum SPF ...,Sun protection,4435.0,6
1351,Sephora Collection,Gradual Self-Tanning Face Gel,Sun protection,4712.0,6
2606,Supergoop!,Everyday Sunscreen Broad Spectrum SPF 50,Sun protection,5266.0,1
2670,Supergoop!,Invincible Setting Powder SPF 45 Refill,Sun protection,5543.0,2
2697,Mdsolarsciences,Mineral Sunscreen Stick Broad Spectrum SPF 40 ...,Sun protection,4989.0,3
2712,Mdsolarsciences,Quick Dry Body Spray With SolSci-X™ Broad Spec...,Sun protection,5543.0,2
2715,Supergoop!,Perfect Day 2-in-1 Everywear Lotion Broad Spec...,Sun protection,5266.0,2
2723,St. Tropez Tanning Essentials,Pro Light Self Tan Bronzing Mist,Sun protection,5543.0,6


In [20]:

# Test sunscreen recommendations using a sunscreen as the reference product.
print("Selected Product:", products.loc[2606, "name"])
sunscreen_results = recommend_products(
    2606, "combination", "Sun protection", 6000
)

print("\nRecommended Sunscreen Alternatives:")
display(sunscreen_results)


Selected Product: Everyday Sunscreen Broad Spectrum SPF 50

Recommended Sunscreen Alternatives:


,brand,name,category,price_pkr,Shared_Actives,Similarity,Match_Type,Cluster
816,innisfree,Daily UV Defense Invisible Broad Spectrum SPF ...,Sun protection,4435.0,0,0.070160,General ingredient similarity,6
2697,Mdsolarsciences,Mineral Sunscreen Stick Broad Spectrum SPF 40 ...,Sun protection,4989.0,0,0.056372,General ingredient similarity,3
2712,Mdsolarsciences,Quick Dry Body Spray With SolSci-X™ Broad Spec...,Sun protection,5543.0,0,0.045145,General ingredient similarity,2
2670,Supergoop!,Invincible Setting Powder SPF 45 Refill,Sun protection,5543.0,0,0.006686,General ingredient similarity,2


In [21]:

# Save the final SkinMatch dataset containing skin-type matches and K-Means clusters.

products.to_csv("skinmatch_recommendations.csv", index=False)

print("Recommendation Dataset Saved Successfully!")
print("Total Products:", len(products))
print("Total Clusters:", products["Cluster"].nunique())
print("Total Skin Types:", len(skin_cols))


Recommendation Dataset Saved Successfully!
Total Products: 2724
Total Clusters: 9
Total Skin Types: 6


### Conclusion

In this phase, the SkinMatch recommendation system was developed by combining the results of KNN classification and K-Means clustering.

The dataset contains **2,724 skincare products**, including six skin-type matching labels and nine ingredient-based clusters.

The recommendation system allows products to be filtered according to:

- **Skin type:** Dry, oily, combination, normal, sensitive, or acne-prone.
- **Product category:** Serum, moisturizer, cleanser, treatment, sun protection, eye care, or mask.
- **Budget:** Maximum price in PKR.

Cosine similarity was used to compare the ingredient lists of skincare products. The recommendation ranking was improved by prioritizing products that share important ingredient features with the selected product.

The system was tested using different skincare products, skin types, categories, and budgets. It successfully generated relevant ingredient-based alternatives and handled cases where no important ingredients were shared.

Products with neither shared key ingredient flags nor positive TF-IDF similarity were excluded from the recommendations.

K-Means cluster numbers were retained for product analysis, while original and KNN-predicted skin-type labels were used for product matching.

**Limitations:** Ingredient similarity does not guarantee equivalent product benefits or skin suitability. Some products have low similarity scores, and the PKR prices are approximate dataset values rather than verified current retail prices.

Finally, the recommendation dataset was saved as **`skinmatch_recommendations.csv`** for integration into the Streamlit application.

This phase successfully completed the recommendation logic for SkinMatch.